# CAE Flight Simulator Manufacturing â€” Solution Installer

This notebook deploys the complete demo into your Fabric workspace.  
It uses [`fabric-cicd`](https://microsoft.github.io/fabric-cicd/) to publish items directly from a GitHub repo.

## Prerequisites
| Requirement | Details |
|---|---|
| Fabric capacity | F2+ (F16+ for AI features) |
| Workspace permissions | Contributor or Admin |

## What gets deployed
| Category | Items |
|---|---|
| **Lakehouse** | `CAEManufacturing_LH` â€” staging area for CSVs |
| **Notebooks** | GetStarted, PostDeploymentConfig, LoadData, 3 Simulators, Agent |

After `fabric-cicd` publishes, the **PostDeploymentConfig** notebook creates the SQL Database tables and loads all data.

**Click Run All to install.**

In [ ]:
# === CONFIGURATION ===
REPO_URL  = "https://github.com/benoit-d/cae-demo.git"   # <-- your repo
REPO_REF  = "master"                                        # tag or branch
WORKSPACE_PATH = "workspace"                              # folder with Fabric items
DATA_PATH      = "data"                                   # folder with CSV seed data

ITEM_TYPES_IN_SCOPE = [
    "Notebook", "Lakehouse", "Environment",
    "Eventhouse",
    "KQLDatabase", "KQLDashboard", "KQLQueryset",
    "SQLDatabase", "DataPipeline",
]
print(f"Install from {REPO_URL} @ {REPO_REF}")

In [ ]:
# Step 1 — Install packages
%pip install --upgrade pip
%pip install -q fabric-cicd azure-identity gitpython

In [ ]:
# Step 2 â€” Clone the repo
import os, shutil, tempfile
from git import Repo

clone_dir = os.path.join(tempfile.gettempdir(), "cae-demo-install")
if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)

print(f"Cloning {REPO_URL} @ {REPO_REF} ...")
Repo.clone_from(REPO_URL, clone_dir, branch=REPO_REF, depth=1)

workspace_dir = os.path.join(clone_dir, WORKSPACE_PATH)
data_dir      = os.path.join(clone_dir, DATA_PATH)
assert os.path.isdir(workspace_dir)
assert os.path.isdir(data_dir)
print("Clone OK")

In [ ]:
# Step 3 â€” Publish Fabric items
from fabric_cicd import FabricWorkspace, publish_all_items

WORKSPACE_ID = ""  # leave blank to auto-detect inside a Fabric notebook

try:
    import notebookutils
    if not WORKSPACE_ID:
        WORKSPACE_ID = notebookutils.fabric.get_workspace_id()
except ImportError:
    if not WORKSPACE_ID:
        raise ValueError("Set WORKSPACE_ID when running outside Fabric.")

try:
    from notebookutils.credentials import getToken
    from azure.core.credentials import AccessToken
    class _FabricCred:
        def get_token(self, *s, **k):
            return AccessToken(getToken("https://api.fabric.microsoft.com"), 0)
    token_credential = _FabricCred()
except ImportError:
    from azure.identity import AzureCliCredential
    token_credential = AzureCliCredential()

ws = FabricWorkspace(
    workspace_id=WORKSPACE_ID,
    repository_directory=workspace_dir,
    item_type_in_scope=ITEM_TYPES_IN_SCOPE,
    token_credential=token_credential,
)
print(f"Publishing to workspace {WORKSPACE_ID} ...")
publish_all_items(ws)
print("All Fabric items published.")

In [ ]:
# Step 4 â€” Upload seed data to the staging Lakehouse
import glob

try:
    import notebookutils
    ws_items = notebookutils.fabric.list_items()
    lh = next((i for i in ws_items if i.get('displayName') == 'CAEManufacturing_LH' and i.get('type') == 'Lakehouse'), None)

    if lh:
        lh_id = lh['id']
        for folder in ['erp', 'hr', 'telemetry', 'plm', 'mes']:
            src = os.path.join(data_dir, folder)
            if not os.path.isdir(src):
                continue
            for f in sorted(glob.glob(os.path.join(src, '*'))):
                fname = os.path.basename(f)
                dest = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
                        f"{lh_id}/Files/data/{folder}/{fname}")
                notebookutils.fs.cp(f"file://{f}", dest)
                print(f"  {folder}/{fname}")
        # Upload KQL scripts
        kql_src = os.path.join(clone_dir, 'scripts', 'kql')
        if os.path.isdir(kql_src):
            for f in sorted(glob.glob(os.path.join(kql_src, '*'))):
                fname = os.path.basename(f)
                dest = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
                        f"{lh_id}/Files/scripts/kql/{fname}")
                notebookutils.fs.cp(f"file://{f}", dest)
                print(f"  scripts/kql/{fname}")

        # Create connections config file (if it doesn't exist)
        import json, requests as _req
        onelake_file = (f"https://onelake.dfs.fabric.microsoft.com/"
                        f"{WORKSPACE_ID}/{lh_id}/Files/config/connections.json")
        storage_token = notebookutils.credentials.getToken("https://storage.azure.com")
        storage_headers = {"Authorization": f"Bearer {storage_token}"}

        check_resp = _req.head(onelake_file, headers=storage_headers)
        if check_resp.status_code == 200:
            print("\nConfig file exists — preserving connection strings")
        else:
            config = {
                "SQL_JDBC_CONNECTION_STRING": "",
                "TELEMETRY_EVENTSTREAM_CONNECTION_STRING": "",
                "CLOCKIN_EVENTSTREAM_CONNECTION_STRING": "",
                "FOUNDRY_AGENT_PROJECT_ENDPOINT": "",
                "FOUNDRY_AGENT_ID": "",
                "TEAMS_WEBHOOK_URL": "",
                "CREATE_ONTOLOGY": "true",
            }
            config_bytes = json.dumps(config, indent=2).encode("utf-8")
            _req.put(f"{onelake_file}?resource=file", headers=storage_headers)
            _req.patch(f"{onelake_file}?action=append&position=0",
                       headers={**storage_headers, "Content-Type": "application/octet-stream"},
                       data=config_bytes)
            _req.patch(f"{onelake_file}?action=flush&position={len(config_bytes)}",
                       headers=storage_headers)
            print("\nCreated config/connections.json — fill in connection strings before running PostDeploymentConfig")
            print("  Edit in Lakehouse > Files > config > connections.json")

        print("\nAll seed data uploaded.")
    else:
        print("CAEManufacturing_LH not found yet. Re-run after it appears.")
except ImportError:
    print("Upload data/ folder to CAEManufacturing_LH/Files/data/ manually.")

In [ ]:
# Step 5 â€” Clean up and next steps
shutil.rmtree(clone_dir, ignore_errors=True)

print("\n" + "=" * 60)
print("  INSTALLATION COMPLETE")
print("=" * 60)
print("\nNext step:")
print("  1. Open Lakehouse > Files > config > connections.json")
print("     Paste your SQL JDBC connection string")
print("  2. Open  PostDeploymentConfig  notebook")
print("  3. Run All \u2014 it creates SQL tables and loads all data")
print("  4. Open  GetStarted  for the guided walkthrough")